# Homework — Working with molecules (RDKit)

Welcome to your first computational-chemistry homework. You'll use **RDKit**, a real
cheminformatics toolkit that pharma and academic labs use, to turn text descriptions of
molecules into objects you can *measure*.

**How this works**
1. Run the **Setup** cell (it installs RDKit — ~30 s the first time).
2. Read each short teaching section and run its example — the expected output is written
   in a `# ...` comment so you can check your run.
3. Answer the numbered **Problems** in the empty cells. About half ask you to *reason*,
   not just compute — read those carefully and write your thinking in comments.
4. Run the **Check in** cell at the end and type its number into the CHEM 438 app.

Run every cell top to bottom. If you skip Setup, nothing else will work.

In [ ]:
# Setup — run this first
!pip install rdkit -q
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
from rdkit.Chem import rdMolDescriptors as rd
print("RDKit is ready.")

## 1. A molecule from text: SMILES

Chemists needed a way to *type* a molecule. A **SMILES string** is that: a compact line
of text that encodes atoms and bonds. A few to recognize:

| SMILES | molecule |
|---|---|
| `O` | water |
| `CCO` | ethanol |
| `c1ccccc1` | benzene |

`Chem.MolFromSmiles(text)` reads a SMILES string and hands back a **molecule object** —
not text, but a thing you can ask questions about (weight, formula, atoms, rings...).

In [ ]:
water   = Chem.MolFromSmiles("O")
ethanol = Chem.MolFromSmiles("CCO")
print(water)     # <rdkit.Chem.rdchem.Mol object at 0x...>  (an object, not the letter O)
print(ethanol)   # <rdkit.Chem.rdchem.Mol object at 0x...>

## 2. The trap: a bad SMILES gives you `None`

RDKit does **not** raise an error when a SMILES string is nonsense. Instead it quietly
returns `None`. That is a trap: your program keeps running, and the crash happens *later*
when you try to measure the `None`. Get used to spotting it now.

In [ ]:
good = Chem.MolFromSmiles("CCO")      # valid
bad  = Chem.MolFromSmiles("CXO")      # 'X' is not a real atom -> nonsense
print(good)    # a Mol object
print(bad)     # None   <-- no error, just None. This is the trap.

## 3. Measuring a molecule

Once you have a molecule object, RDKit measures it for you. The four you'll use most:

- `Descriptors.MolWt(mol)` — molecular weight (g/mol)
- `rd.CalcMolFormula(mol)` — chemical formula as a string
- `mol.GetNumAtoms()` — number of atoms (heavy atoms; H's are implicit)
- `rd.CalcNumRings(mol)` — number of rings

Notice RDKit counts **heavy atoms** only (C, N, O...). Hydrogens are stored implicitly,
so `CCO` reports 3 atoms, not 9.

In [ ]:
ethanol = Chem.MolFromSmiles("CCO")
print("formula:", rd.CalcMolFormula(ethanol))          # C2H6O
print("weight :", round(Descriptors.MolWt(ethanol), 1))# 46.1
print("atoms  :", ethanol.GetNumAtoms())               # 3   (heavy atoms only)
print("rings  :", rd.CalcNumRings(ethanol))            # 0

## 4. Drawing a molecule

A formula tells you *what*; a picture tells you *how it's connected*. In Colab, just make
the image the **last thing** in a cell and it displays automatically.

- `Draw.MolToImage(mol)` — one molecule
- `Draw.MolsToGridImage([m1, m2], legends=["a", "b"])` — several in a grid

In [ ]:
caffeine = Chem.MolFromSmiles("CN1C=NC2=C1C(=O)N(C(=O)N2C)C")
aspirin  = Chem.MolFromSmiles("CC(=O)OC1=CC=CC=C1C(=O)O")

# In Colab, leaving the image as the last line shows it.
Draw.MolsToGridImage([caffeine, aspirin], legends=["caffeine", "aspirin"],
                     molsPerRow=2, subImgSize=(250, 200))

## 5. Hydrogen-bond donors/acceptors, and substructure search

**H-bond donors and acceptors** control how a molecule interacts with water, proteins, and
other molecules — they're central to why drugs dissolve and bind.

- `Descriptors.NumHDonors(mol)` — groups that can *donate* an H-bond (like O–H, N–H)
- `Descriptors.NumHAcceptors(mol)` — atoms that can *accept* one (lone-pair O and N)

**Substructure search** answers "does this molecule *contain* a particular group?" You
build a query pattern with `Chem.MolFromSmarts(...)` and test with `mol.HasSubstructMatch(query)`.

In [ ]:
ethanol = Chem.MolFromSmiles("CCO")
print("donors   :", Descriptors.NumHDonors(ethanol))    # 1  (the O-H)
print("acceptors:", Descriptors.NumHAcceptors(ethanol)) # 1  (the O)

# Does ethanol contain an alcohol (C-O-H)?  Pattern: a carbon single-bonded to an -OH.
alcohol = Chem.MolFromSmarts("[CX4][OX2H]")
print("has -OH  :", ethanol.HasSubstructMatch(alcohol)) # True

## Problems

Write your code in the empty cell under each problem. Some problems ask you to **reason** —
write your guess or explanation in `#` comments, then run code to check it.

**Problem 1.** Aspirin has the SMILES `CC(=O)OC1=CC=CC=C1C(=O)O`.
Make the molecule, then print its **chemical formula** and its **molecular weight** rounded
to 1 decimal place.

**Problem 2 (predict first).** Glucose — table sugar's cousin — has the SMILES
`C(C1C(C(C(C(O1)O)O)O)O)O`. Caffeine's SMILES is `CN1C=NC2=C1C(=O)N(C(=O)N2C)C`.

**First write down your guess** (in a comment): which is heavier, glucose or caffeine?
*Then* compute both molecular weights and compare. Were you right? In a comment, say what
in each structure explains the answer (hint: caffeine carries four nitrogens).

**Problem 3 (read the trap).** A classmate typos glucose's SMILES as `C(C1C(C(C(C(O1)O)O)O)O)Q`
(note the stray `Q`). Run `mol = Chem.MolFromSmiles("...")` on that typo and `print(mol)`.

Now try `Descriptors.MolWt(mol)` on it. **Read the error message** and, in a comment,
answer: what value did `MolFromSmiles` return for the bad SMILES, and *why* does calling
`MolWt` on it fail? What one line could you add to catch this before it crashes?

**Problem 4 (interpret).** Compute caffeine's H-bond donors and acceptors. You should find
**0 donors and 6 acceptors**.

Answer the multiple-choice below by printing your letter (e.g. `print("B")`), and in a
comment explain your reasoning:

> Caffeine has 0 H-bond donors. Compared with a molecule that has several O–H/N–H donors,
> caffeine is most likely to...
>
> **A.** form more hydrogen bonds *to itself*, making it a hard solid
> **B.** rely on its *acceptor* atoms to interact with water, and pass through fatty cell
>        membranes fairly easily
> **C.** be completely insoluble in everything
> **D.** have no way to interact with water at all

Then, in a comment: what might 0 donors + 6 acceptors suggest about how caffeine dissolves
and how it gets into cells?

**Problem 5 (compare, then search).**

**(a)** Ethanol is `CCO` and dimethyl ether is `COC`. Print the **formula and molecular
weight of each**. They have the *same formula* but different structures (they're isomers).
Before running: do you expect the same molecular weight? Explain in a comment why the MW is
or isn't the same, even though the structures differ.

**(b)** Build a carboxylic-acid query with `Chem.MolFromSmarts("C(=O)[OX2H1]")`. Use
`HasSubstructMatch` to test whether **ibuprofen** (`CC(C)Cc1ccc(cc1)C(C)C(=O)O`) and
**benzene** (`c1ccccc1`) each contain a carboxylic acid. Explain the two results in a
comment.

**(c) (why).** In a comment: why is a SMILES string a better description of a molecule than
just the formula `C8H10N4O2`? (Think about what Problem 5a just showed you.)

## Check in

Run the cell below. It prints one number — the molecular weight of **caffeine**, rounded to
1 decimal place. **Type that number into the CHEM 438 app to mark this homework complete.**

In [ ]:
caffeine = Chem.MolFromSmiles("CN1C=NC2=C1C(=O)N(C(=O)N2C)C")
print("Your check-in value:", round(Descriptors.MolWt(caffeine), 1))